# RAG Module · Notebook 3 · The RAG Pipeline
Build a complete RAG system with ChromaDB — indexing, retrieval, and grounded generation.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
EMBED_MODEL = 'gemini-embedding-001'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('✅ Setup complete. Model:', MODEL, '| Embed:', EMBED_MODEL, '| Retry-on-429: enabled')

✅ Setup complete. Model: gemma-4-31b-it | Embed: gemini-embedding-001 | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 1. ChromaDB — A Vector Database for RAG
ChromaDB stores text chunks alongside their embeddings and retrieves the most relevant ones instantly.

**Developer analogy:** A regular database stores rows and lets you query by column value. ChromaDB stores embeddings and lets you query by meaning — 'find me the 3 most relevant chunks for this question.'

What ChromaDB handles automatically:
- Storing vectors alongside document text
- Nearest-neighbor search (finds semantically similar chunks fast)
- Persistent storage — we use in-memory mode for this course

Key API surface:
```
collection.add(ids, documents, embeddings)   → store chunks
collection.query(query_embeddings, n_results) → retrieve top-K
collection.count()                            → how many chunks stored
```

In [4]:
import chromadb

def embed(text: str) -> list:
    """Embed a string using the Gemini embedding model."""
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

# Create an in-memory ChromaDB client and a collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

print("✅ ChromaDB ready. Collection:", collection.name)

✅ ChromaDB ready. Collection: rag_demo


In [5]:
# Manually add a few documents to get a feel for the API
demo_docs = [
    "Python was created by Guido van Rossum and released in 1991.",
    "Django is a high-level Python web framework for rapid development.",
    "NumPy provides fast array operations for numerical computing in Python.",
]

for i, doc in enumerate(demo_docs):
    collection.add(
        ids=[f"doc_{i}"],
        documents=[doc],
        embeddings=[embed(doc)]
    )

print(f"Stored {collection.count()} documents.\n")

# Query the collection
query = "what tools exist for Python web development?"
query_embedding = embed(query)

results = collection.query(query_embeddings=[query_embedding], n_results=2)

print(f"Query: '{query}'\n")
for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0]), 1):
    print(f"#{i} (distance={dist:.4f}): {doc}")

print("\nNote: lower distance = more similar in ChromaDB (uses L2 distance by default).")

Stored 3 documents.



Query: 'what tools exist for Python web development?'

#1 (distance=0.6184): Django is a high-level Python web framework for rapid development.
#2 (distance=0.7772): Python was created by Guido van Rossum and released in 1991.

Note: lower distance = more similar in ChromaDB (uses L2 distance by default).


| ChromaDB concept | What it does |
|---|---|
| `collection.add(ids, documents, embeddings)` | Store chunk text + its embedding vector |
| `collection.query(query_embeddings, n_results)` | Find the n closest chunks by vector distance |
| `collection.count()` | Return number of stored chunks |
| `chromadb.Client()` | In-memory client (no disk, resets on restart) |
| `chromadb.PersistentClient(path)` | Persist to disk (for production use) |

---
## 2. Indexing — Chunk, Embed, and Store
Before querying, we split all documents into chunks and store them in ChromaDB with their embeddings.

This is a one-time operation (or run when documents change). The corpus used here is self-contained — no external downloads needed.

In [6]:
def chunk_overlap(text: str, chunk_size: int = 300, overlap: int = 60) -> list:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]  # drop tiny tail chunks

# Self-contained corpus: 12 documents about Python
CORPUS = {
    "python_history": """Python is a high-level, general-purpose programming language created by Guido van Rossum.
Development began in the late 1980s and the first version was released in 1991.
The name Python was inspired by the British comedy group Monty Python, not the snake.
Guido van Rossum served as Python's Benevolent Dictator For Life (BDFL) until 2018.
Python 2 was released in 2000 and Python 3 in 2008. Python 2 reached end-of-life in January 2020.""",

    "python_philosophy": """Python's design philosophy is captured in the Zen of Python (PEP 20).
Key principles include: Beautiful is better than ugly. Explicit is better than implicit.
Simple is better than complex. Readability counts. There should be one obvious way to do it.
Python emphasizes code readability and developer productivity over raw execution speed.
The language uses significant whitespace (indentation) to delimit code blocks.""",

    "python_data_types": """Python has several built-in data types: integers, floats, strings, booleans, and None.
Collection types include lists (mutable, ordered), tuples (immutable, ordered),
sets (unordered, unique elements), and dictionaries (key-value pairs).
Python uses dynamic typing — variables do not need type declarations.
Type hints were introduced in Python 3.5 via PEP 484 for optional static analysis.""",

    "python_web": """Django is a high-level Python web framework that encourages rapid development.
Flask is a lightweight micro-framework for building web applications in Python.
FastAPI is a modern, fast web framework for building APIs with Python type hints.
Python web frameworks follow WSGI or ASGI standards for server communication.
Django includes an ORM, admin interface, authentication, and many built-in features.""",

    "python_data_science": """NumPy provides efficient array operations and mathematical functions for Python.
Pandas offers data structures (DataFrame, Series) for data manipulation and analysis.
Matplotlib and Seaborn are popular Python libraries for data visualization.
Scikit-learn provides machine learning algorithms for classification, regression, and clustering.
Jupyter notebooks are widely used in data science for interactive Python code and visualization.""",

    "python_ml": """TensorFlow is an open-source machine learning framework developed by Google.
PyTorch is a machine learning framework developed by Meta, popular in research.
Both TensorFlow and PyTorch support GPU acceleration via CUDA for faster training.
Hugging Face Transformers provides pre-trained models for NLP tasks in Python.
Python is the dominant language for machine learning and artificial intelligence.""",

    "python_packaging": """pip is the standard package manager for Python, installing packages from PyPI.
PyPI (Python Package Index) hosts hundreds of thousands of third-party packages.
Virtual environments (venv, conda) isolate project dependencies to avoid conflicts.
pyproject.toml is the modern standard for Python project configuration (PEP 621).
Poetry and uv are modern alternatives to pip for dependency management.""",

    "python_async": """Python's asyncio library enables asynchronous programming with async/await syntax.
The Global Interpreter Lock (GIL) limits true CPU parallelism in CPython.
asyncio is ideal for I/O-bound tasks (network requests, file operations).
For CPU-bound tasks, the multiprocessing module bypasses the GIL using separate processes.
Python 3.12 introduced per-interpreter GIL, enabling true thread parallelism in subinterpreters.""",

    "python_testing": """pytest is the most popular testing framework for Python applications.
unittest is Python's built-in testing framework, part of the standard library.
Test coverage is measured with the coverage.py tool, often integrated with pytest.
Mocking in Python tests uses unittest.mock or the pytest-mock plugin.
TDD (Test-Driven Development) is widely practised in the Python community.""",

    "python_performance": """Python is slower than compiled languages like C, C++, or Rust due to interpretation.
List comprehensions and generator expressions are faster than equivalent for-loops.
The bisect module provides binary search for sorted lists in O(log n) time.
CPython is the reference implementation; PyPy offers a JIT compiler for significant speedups.
Profiling tools like cProfile and line_profiler help identify bottlenecks.""",

    "python_typing": """Python 3.5 introduced type hints with PEP 484. Type hints are optional annotations.
mypy is the most popular static type checker for Python.
The typing module provides generic types like List, Dict, Optional, and Union.
Python 3.10 added the union operator syntax: int | str instead of Union[int, str].
Dataclasses (Python 3.7+) generate __init__, __repr__, and other methods from type annotations.""",

    "python_standard_library": """Python's standard library is described as 'batteries included' — it covers a huge range.
Key modules: os (file system), sys (interpreter), json (JSON parsing), re (regex),
datetime (dates/times), pathlib (paths), collections (deque, Counter, defaultdict),
itertools (combinatorial iterators), functools (higher-order functions).
The standard library is always available — no pip install required.""",
}

print(f"Corpus: {len(CORPUS)} documents")
for key, text in CORPUS.items():
    print(f"  {key}: {len(text)} chars")

Corpus: 12 documents
  python_history: 437 chars
  python_philosophy: 418 chars
  python_data_types: 390 chars
  python_web: 403 chars
  python_data_science: 437 chars
  python_ml: 400 chars
  python_packaging: 397 chars
  python_async: 418 chars
  python_testing: 376 chars
  python_performance: 413 chars
  python_typing: 398 chars
  python_standard_library: 396 chars


In [7]:
# Fresh collection for indexing
collection = chroma_client.get_or_create_collection(name="python_knowledge")
# Clear if already exists from a previous run
if collection.count() > 0:
    chroma_client.delete_collection("python_knowledge")
    collection = chroma_client.create_collection("python_knowledge")

def index_documents(corpus: dict, collection, chunk_size: int = 300, overlap: int = 60):
    """Chunk all documents, embed each chunk, and add to ChromaDB."""
    chunk_id = 0
    for doc_name, text in corpus.items():
        chunks = chunk_overlap(text, chunk_size=chunk_size, overlap=overlap)
        for chunk_text in chunks:
            embedding = embed(chunk_text)
            collection.add(
                ids=[f"{doc_name}_{chunk_id}"],
                documents=[chunk_text],
                embeddings=[embedding],
                metadatas=[{"source": doc_name}]
            )
            chunk_id += 1
    return chunk_id

print("Indexing corpus into ChromaDB...")
total = index_documents(CORPUS, collection)
print(f"✅ Indexed {total} chunks from {len(CORPUS)} documents.")
print(f"   Collection count: {collection.count()}")

Indexing corpus into ChromaDB...


✅ Indexed 24 chunks from 12 documents.
   Collection count: 24


In [8]:
# ✏️ Add your own document to the knowledge base
my_doc_name = "my_custom_doc"
my_doc_text = """Replace this with your own text. It will be chunked, embedded,
and added to the ChromaDB collection so you can query it below."""

my_chunks = chunk_overlap(my_doc_text)
for i, chunk_text in enumerate(my_chunks):
    collection.add(
        ids=[f"{my_doc_name}_{i}"],
        documents=[chunk_text],
        embeddings=[embed(chunk_text)],
        metadatas=[{"source": my_doc_name}]
    )

print(f"Added {len(my_chunks)} chunks from '{my_doc_name}'.")
print(f"Total chunks in collection: {collection.count()}")

Added 1 chunks from 'my_custom_doc'.
Total chunks in collection: 25


---
## 3. Retrieval — Finding the Right Chunks
Given a user's question, find the top-K most relevant chunks from ChromaDB.

The retrieval step is what makes RAG efficient — instead of sending all documents, we send only the most relevant 2–5 chunks.

In [9]:
def retrieve(question: str, collection, top_k: int = 3) -> list:
    """
    Embed the question, query ChromaDB for top_k closest chunks.
    Returns list of (distance, chunk_text, source) tuples.
    """
    q_embedding = embed(question)
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        include=['documents', 'distances', 'metadatas']
    )
    chunks = []
    for doc, dist, meta in zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    ):
        chunks.append({
            'text': doc,
            'distance': dist,
            'source': meta.get('source', 'unknown')
        })
    return chunks

# Run 3 test queries and inspect retrieved chunks
test_queries = [
    "Who created Python and when was it released?",
    "How do I build a web API in Python?",
    "What is Python's approach to concurrency and the GIL?",
]

for query in test_queries:
    chunks = retrieve(query, collection, top_k=3)
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    for i, chunk in enumerate(chunks, 1):
        print(f"  #{i} [source={chunk['source']}, dist={chunk['distance']:.4f}]")
        print(f"      {chunk['text'][:100]}...")


Query: 'Who created Python and when was it released?'
------------------------------------------------------------
  #1 [source=python_history, dist=0.4579]
      Python is a high-level, general-purpose programming language created by Guido van Rossum.
Developmen...
  #2 [source=python_history, dist=0.5237]
      not the snake.
Guido van Rossum served as Python's Benevolent Dictator For Life (BDFL) until 2018.
P...
  #3 [source=python_philosophy, dist=0.7171]
      Python's design philosophy is captured in the Zen of Python (PEP 20).
Key principles include: Beauti...



Query: 'How do I build a web API in Python?'
------------------------------------------------------------
  #1 [source=python_web, dist=0.5461]
      Django is a high-level Python web framework that encourages rapid development.
Flask is a lightweigh...
  #2 [source=python_web, dist=0.6309]
      Python web frameworks follow WSGI or ASGI standards for server communication.
Django includes an ORM...
  #3 [source=python_philosophy, dist=0.7630]
      Python's design philosophy is captured in the Zen of Python (PEP 20).
Key principles include: Beauti...



Query: 'What is Python's approach to concurrency and the GIL?'
------------------------------------------------------------
  #1 [source=python_async, dist=0.4694]
      Python's asyncio library enables asynchronous programming with async/await syntax.
The Global Interp...
  #2 [source=python_async, dist=0.4719]
      ound tasks, the multiprocessing module bypasses the GIL using separate processes.
Python 3.12 introd...
  #3 [source=python_performance, dist=0.7253]
      Python is slower than compiled languages like C, C++, or Rust due to interpretation.
List comprehens...


| Retrieval concept | What it does |
|---|---|
| `embed(question)` | Convert the user's question into a vector |
| `collection.query(query_embeddings, n_results)` | Find the n chunks whose vectors are closest to the question vector |
| Distance score | Lower = more similar; threshold ~0.5 typically means a good match |
| `top_k` | How many chunks to retrieve; 3–5 is typical for RAG |
| Metadata filtering | ChromaDB supports filtering by metadata fields alongside vector search |

---
## 4. Augmented Generation — Answering from Evidence
Inject retrieved chunks as context and instruct the model to answer using only that evidence.

This is the 'A' (Augmented) in RAG — we augment the prompt with retrieved facts before generating.

The grounding instruction prevents hallucination beyond what's in the retrieved context.

In [10]:
def rag_pipeline(question: str, collection, top_k: int = 3) -> dict:
    """
    Full RAG pipeline:
    1. Retrieve top-K relevant chunks from ChromaDB
    2. Build an augmented prompt with retrieved context
    3. Generate a grounded answer
    Returns dict with answer, retrieved chunks, and their distances.
    """
    # Step 1: Retrieve
    chunks = retrieve(question, collection, top_k=top_k)

    # Step 2: Build context block
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(f"[Source {i}: {chunk['source']}]\n{chunk['text']}")
    context = "\n\n".join(context_parts)

    # Step 3: Augmented prompt
    prompt = f"""Answer the question using ONLY the context provided below.
If the answer is not in the context, respond with exactly:
"I don't have that information in my knowledge base."

Context:
{context}

Question: {question}

Answer:"""

    # Step 4: Generate
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=cfg(temperature=0.0)
    )

    return {
        'answer': get_text(r),
        'chunks': chunks,
    }

In [11]:
questions = [
    # Well-covered by corpus
    ("In-scope", "Who invented Python and what was it named after?"),
    # Partially covered
    ("Partial",  "What are the best practices for testing Python code?"),
    # Not in corpus at all
    ("Out-of-scope", "What is the best Python library for building mobile apps?"),
]

for label, question in questions:
    print(f"\n{'='*60}")
    print(f"[{label}] {question}")
    print('='*60)

    result = rag_pipeline(question, collection)

    print(f"\n📄 Retrieved chunks:")
    for i, chunk in enumerate(result['chunks'], 1):
        print(f"  #{i} [{chunk['source']}, dist={chunk['distance']:.4f}]: {chunk['text'][:80]}...")

    print(f"\n💬 Answer:")
    print(result['answer'])


[In-scope] Who invented Python and what was it named after?



📄 Retrieved chunks:
  #1 [python_history, dist=0.5032]: Python is a high-level, general-purpose programming language created by Guido va...
  #2 [python_history, dist=0.6283]: not the snake.
Guido van Rossum served as Python's Benevolent Dictator For Life ...
  #3 [python_philosophy, dist=0.7074]: Python's design philosophy is captured in the Zen of Python (PEP 20).
Key princi...

💬 Answer:
Python was created by Guido van Rossum and was named after the British comedy group Monty Python.

[Partial] What are the best practices for testing Python code?



📄 Retrieved chunks:
  #1 [python_testing, dist=0.4716]: in Python tests uses unittest.mock or the pytest-mock plugin.
TDD (Test-Driven D...
  #2 [python_testing, dist=0.5211]: pytest is the most popular testing framework for Python applications.
unittest i...
  #3 [python_philosophy, dist=0.6864]: Python's design philosophy is captured in the Zen of Python (PEP 20).
Key princi...

💬 Answer:
TDD (Test-Driven Development) is widely practised in the Python community. pytest is the most popular testing framework, while unittest is the built-in testing framework. Test coverage is measured with the coverage.py tool, often integrated with pytest, and mocking uses unittest.mock or the pytest-mock plugin.

[Out-of-scope] What is the best Python library for building mobile apps?



📄 Retrieved chunks:
  #1 [python_web, dist=0.7235]: Django is a high-level Python web framework that encourages rapid development.
F...
  #2 [python_testing, dist=0.7388]: pytest is the most popular testing framework for Python applications.
unittest i...
  #3 [python_web, dist=0.7389]: Python web frameworks follow WSGI or ASGI standards for server communication.
Dj...

💬 Answer:
I don't have that information in my knowledge base.


| RAG step | What happens |
|---|---|
| Retrieve | Embed the question; query ChromaDB for top-K closest chunks |
| Build context | Format the retrieved chunks into a readable context block |
| Augmented prompt | Prepend context to the question before sending to the LLM |
| Grounding instruction | "Answer using ONLY the context" — constrains the model to evidence |
| Out-of-scope handling | Model returns a fixed fallback string when context has no answer |

---
## 5. RAG vs. No RAG — Why It Matters
Compare a bare LLM answer against a RAG answer, and measure the token cost difference.

In [12]:
comparison_question = "What year was Python created and who created it?"

print("=" * 60)
print("WITHOUT RAG (bare LLM call):")
print("=" * 60)
r_bare = client.models.generate_content(
    model=MODEL,
    contents=comparison_question,
    config=cfg(temperature=0.0)
)
print(get_text(r_bare))

print("\n" + "=" * 60)
print("WITH RAG (grounded on retrieved chunks):")
print("=" * 60)
result = rag_pipeline(comparison_question, collection)
print(result['answer'])
print(f"\n(Retrieved from: {', '.join(c['source'] for c in result['chunks'])})") 

WITHOUT RAG (bare LLM call):


Python was created by **Guido van Rossum** and was first released in **1991**.

WITH RAG (grounded on retrieved chunks):


Python was created by Guido van Rossum and the first version was released in 1991.

(Retrieved from: python_history, python_history, python_philosophy)


In [13]:
# Compare token cost: stuffing all documents vs. RAG top-3 chunks
all_text = "\n\n".join(CORPUS.values())
top3_text = "\n\n".join(c['text'] for c in retrieve(comparison_question, collection, top_k=3))

all_tokens  = client.models.count_tokens(model=MODEL, contents=all_text).total_tokens
top3_tokens = client.models.count_tokens(model=MODEL, contents=top3_text).total_tokens

print(f"Full corpus stuffed into prompt: {all_tokens:,} tokens")
print(f"RAG top-3 chunks:                {top3_tokens:,} tokens")
print(f"Token savings:                   {all_tokens - top3_tokens:,} tokens ({(1 - top3_tokens/all_tokens)*100:.1f}% reduction)")
print(f"\nRAG uses {all_tokens // top3_tokens}x fewer input tokens per query.")

Full corpus stuffed into prompt: 1,073 tokens
RAG top-3 chunks:                213 tokens
Token savings:                   860 tokens (80.1% reduction)

RAG uses 5x fewer input tokens per query.


| | Bare LLM | Full-context stuffing | RAG |
|---|---|---|---|
| Knows private data | No | Yes (if fits) | Yes |
| Scales to large corpus | N/A | No | Yes |
| Token cost per query | Low | High | Low |
| Hallucination risk | High | Low | Low |
| Up-to-date knowledge | No | Yes | Yes |

---
### Key Takeaways

| Concept | One-liner |
|---|---|
| ChromaDB | Vector database — stores chunks + embeddings, retrieves by similarity |
| Indexing | Chunk → embed → `collection.add()` — done once per corpus |
| Retrieval | `collection.query()` — finds top-K chunks closest to the question |
| Augmentation | Inject retrieved chunks as context before generating |
| Grounding prompt | "Answer using ONLY the context below" — prevents hallucination |
| Token savings | RAG sends ~3 chunks instead of the whole corpus |

**Next:** Notebook 4 covers RAG evaluation — how to measure whether your system is actually working.